# DiffiScape Solver Benchmark: Torch vs JAX vs AMGX

Comprehensive benchmark comparing the Torch (CuPy/scipy) and JAX (JAXScape/AMJax) circuit
solver backends across:

- **Two absorption strategies**: Uniform vs Boundary-only
- **Five solver backends**: Torch CPU (pyamg), Torch GPU (Jacobi), Torch GPU (AMGX), JAX CPU (AMJax), JAX GPU (AMJax)
- **Three grid sizes**: 100×100, 500×500, 1000×1000

## Hardware

- **GPU**: NVIDIA GeForce RTX 4070 Ti SUPER (16 GB VRAM)
- **CPU**: AMD Ryzen 7 7800X3D (8-core)
- **Platform**: WSL2 Ubuntu on Windows 11, CUDA 12.4 (driver 13.2)

## Absorption Strategies

| Strategy | System | Condition | Use Case |
|---|---|---|---|
| **Uniform** | $(L + \alpha I)v = \mathbf{1}$ | Well-conditioned (diagonal dominance everywhere) | Fast approximation, no boundary artifacts |
| **Boundary** | $(L + \alpha I_\partial)v = \mathbf{1}_{\text{int}}$ | Ill-conditioned (interior has no diagonal boost) | Ecologically correct boundary grounding |

## Edge Weighting Difference

- **Torch**: Harmonic mean of resistance $w_{ij} = 2/(R_i + R_j)$
- **JAX**: Arithmetic mean of permeability $w_{ij} = (p_i + p_j)/2$

These produce correlated but not identical current density fields (Pearson r ≈ 0.71 for uniform, r ≈ 0.9995 for boundary).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# Benchmark results (RTX 4070 Ti SUPER, Ryzen 7 7800X3D, WSL2, CUDA 12.4)
# Each value: mean_ms over 3 runs (warmup excluded)
results = {
    "100x100": {
        "uniform": {
            "Torch CPU\n(pyamg)": 10.7,
            "Torch GPU\n(Jacobi)": 4.0,
            "Torch GPU\n(AMGX)": 51.6,
            "JAX CPU\n(AMJax)": 14.6,
            "JAX GPU\n(AMJax)": 74.0,
        },
        "boundary": {
            "Torch CPU\n(pyamg)": 24.7,
            "Torch GPU\n(Jacobi)": 136.4,
            "Torch GPU\n(AMGX)": 2412.7,
            "JAX CPU\n(AMJax)": 19.5,
            "JAX GPU\n(AMJax)": 62.9,
        },
    },
    "500x500": {
        "uniform": {
            "Torch CPU\n(pyamg)": 345.7,
            "Torch GPU\n(Jacobi)": 22.5,
            "Torch GPU\n(AMGX)": 171.4,
            "JAX CPU\n(AMJax)": 341.8,
            "JAX GPU\n(AMJax)": 33.3,
        },
        "boundary": {
            "Torch CPU\n(pyamg)": 1043.4,
            "Torch GPU\n(Jacobi)": 621.3,
            "Torch GPU\n(AMGX)": 7218.7,
            "JAX CPU\n(AMJax)": 545.4,
            "JAX GPU\n(AMJax)": 51.1,
        },
    },
    "1000x1000": {
        "uniform": {
            "Torch CPU\n(pyamg)": 1597.1,
            "Torch GPU\n(Jacobi)": 29.2,
            "Torch GPU\n(AMGX)": 553.2,
            "JAX CPU\n(AMJax)": 1629.6,
            "JAX GPU\n(AMJax)": 130.5,
        },
        "boundary": {
            "Torch CPU\n(pyamg)": 3907.6,
            "Torch GPU\n(Jacobi)": 1026.1,
            "Torch GPU\n(AMGX)": 13961.3,
            "JAX CPU\n(AMJax)": 2673.0,
            "JAX GPU\n(AMJax)": 222.0,
        },
    },
}

backends = list(results["100x100"]["uniform"].keys())
sizes = ["100x100", "500x500", "1000x1000"]
colors = ["#4e79a7", "#59a14f", "#e15759", "#76b7b2", "#f28e2b"]

## 1. Uniform Absorption: $(L + \alpha I)v = \mathbf{1}$

All nodes get absorption $\alpha = 0.01$ on the diagonal, making the system well-conditioned.
Jacobi preconditioning is near-optimal here because every diagonal entry is boosted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle("Uniform Absorption: (L + αI)v = 1", fontsize=14, fontweight="bold")

for i, size in enumerate(sizes):
    ax = axes[i]
    vals = [results[size]["uniform"][b] for b in backends]
    bars = ax.bar(range(len(backends)), vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_title(f"{size}\n({int(size.split('x')[0])**2:,} cells)")
    ax.set_xticks(range(len(backends)))
    ax.set_xticklabels(backends, fontsize=8)
    ax.set_ylabel("Time (ms)" if i == 0 else "")
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.yaxis.get_major_formatter().set_scientific(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                f"{val:.0f}" if val >= 10 else f"{val:.1f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")

fig.tight_layout()
plt.savefig("uniform_absorption.png", dpi=150, bbox_inches="tight")
plt.show()

### Uniform Absorption Results

| Grid | Torch CPU (pyamg) | Torch GPU (Jacobi) | Torch GPU (AMGX) | JAX CPU (AMJax) | JAX GPU (AMJax) |
|---|---|---|---|---|---|
| 100×100 | 10.7 ms | **4.0 ms** | 51.6 ms | 14.6 ms | 74.0 ms |
| 500×500 | 345.7 ms | **22.5 ms** | 171.4 ms | 341.8 ms | 33.3 ms |
| 1000×1000 | 1597.1 ms | **29.2 ms** | 553.2 ms | 1629.6 ms | 130.5 ms |

**Winner: Torch GPU (Jacobi)** — 55× faster than CPU at 1000×1000.

Jacobi preconditioning is near-optimal for uniform absorption because every diagonal entry
has the absorption boost $\alpha$. The system is well-conditioned (condition number ~ $O(\alpha^{-1})$),
so CG converges in very few iterations.

JAX GPU is 4.5× slower than Torch GPU Jacobi but still 12× faster than CPU.
AMGX's AMG hierarchy setup overhead isn't worth it when Jacobi already converges fast.

## 2. Boundary Absorption: $(L + \alpha I_\partial)v = \mathbf{1}_{\text{int}}$

Only boundary nodes get absorption. Interior nodes have pure Laplacian rows (zero diagonal boost),
making the system ill-conditioned. Sources inject at interior nodes only.

This is the ecologically meaningful strategy: current flows from the landscape interior and
drains through the boundary, mimicking organism dispersal with landscape edges as sinks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle("Boundary Absorption: (L + αI∂)v = 1_int", fontsize=14, fontweight="bold")

for i, size in enumerate(sizes):
    ax = axes[i]
    vals = [results[size]["boundary"][b] for b in backends]
    bars = ax.bar(range(len(backends)), vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_title(f"{size}\n({int(size.split('x')[0])**2:,} cells)")
    ax.set_xticks(range(len(backends)))
    ax.set_xticklabels(backends, fontsize=8)
    ax.set_ylabel("Time (ms)" if i == 0 else "")
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.yaxis.get_major_formatter().set_scientific(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                f"{val:.0f}" if val >= 10 else f"{val:.1f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")

fig.tight_layout()
plt.savefig("boundary_absorption.png", dpi=150, bbox_inches="tight")
plt.show()

### Boundary Absorption Results

| Grid | Torch CPU (pyamg) | Torch GPU (Jacobi) | Torch GPU (AMGX) | JAX CPU (AMJax) | JAX GPU (AMJax) |
|---|---|---|---|---|---|
| 100×100 | 24.7 ms | 136.4 ms | 2412.7 ms | **19.5 ms** | 62.9 ms |
| 500×500 | 1043.4 ms | 621.3 ms | 7218.7 ms | 545.4 ms | **51.1 ms** |
| 1000×1000 | 3907.6 ms | 1026.1 ms | 13961.3 ms | 2673.0 ms | **222.0 ms** |

**Winner: JAX GPU (AMJax)** — 4.6× faster than Torch GPU Jacobi, 18× faster than Torch CPU at 1000×1000.

The boundary absorption system exposes the fundamental weakness of Jacobi preconditioning:
interior nodes have no diagonal boost, so Jacobi's $M = \text{diag}(L)^{-1}$ provides no
acceleration for the bulk of the matrix. CG requires many more iterations to converge.

**AMGX is catastrophically slow** (14 seconds at 1000×1000). AMGX's aggregation-based AMG
takes ~2000 iterations because its coarsening strategy doesn't capture the boundary-only
grounding structure. Each iteration is individually fast on GPU (~7ms), but the iteration
count explodes compared to the well-conditioned uniform case.

JAX's AMJax solver (algebraic multigrid in JAX) handles this correctly — its AMG hierarchy
captures the boundary grounding structure, converging in far fewer iterations.

## 3. Strategy Comparison: Uniform vs Boundary Slowdown

How much slower is boundary absorption relative to uniform for each backend?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Boundary / Uniform Slowdown Factor (1000×1000)", fontsize=14, fontweight="bold")

size = "1000x1000"
slowdowns = []
labels_clean = ["Torch CPU\n(pyamg)", "Torch GPU\n(Jacobi)", "Torch GPU\n(AMGX)", "JAX CPU\n(AMJax)", "JAX GPU\n(AMJax)"]
for b in backends:
    slowdowns.append(results[size]["boundary"][b] / results[size]["uniform"][b])

bars = ax.bar(range(len(backends)), slowdowns, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xticks(range(len(backends)))
ax.set_xticklabels(labels_clean, fontsize=9)
ax.set_ylabel("Slowdown factor (boundary / uniform)")
ax.axhline(y=1, color="gray", linestyle="--", alpha=0.5)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

for bar, val in zip(bars, slowdowns):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f"{val:.1f}×", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("slowdown_factor.png", dpi=150, bbox_inches="tight")
plt.show()

print("Slowdown factors (boundary / uniform) at 1000×1000:")
for b, s in zip(labels_clean, slowdowns):
    print(f"  {b.replace(chr(10), ' '):25s} {s:6.1f}×")

## 4. GPU Scaling Comparison

How each GPU backend scales with problem size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
n_cells = [10_000, 250_000, 1_000_000]

gpu_backends = [
    ("Torch GPU\n(Jacobi)", "#59a14f", "s"),
    ("Torch GPU\n(AMGX)", "#e15759", "^"),
    ("JAX GPU\n(AMJax)", "#f28e2b", "o"),
]

for ax, strategy, title in zip(axes, ["uniform", "boundary"],
                                ["Uniform Absorption", "Boundary Absorption"]):
    for name, color, marker in gpu_backends:
        vals = [results[s][strategy][name] for s in sizes]
        label = name.replace("\n", " ")
        ax.plot(n_cells, vals, marker=marker, color=color, linewidth=2,
                markersize=8, label=label)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Grid cells")
    ax.set_ylabel("Time (ms)")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f"{int(x):,}"))

fig.suptitle("GPU Solver Scaling", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("gpu_scaling.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Gradient Verification

AMGX adjoint solve produces identical gradients to the CPU reference (Dirichlet boundary mode).

| Metric | Value |
|---|---|
| Pearson correlation (CPU vs AMGX gradient) | 1.0000000000 |
| Max relative error | 1.40e-9 |

Cross-framework correlation (voltage fields, 100×100):

| Strategy | Torch vs JAX Pearson r |
|---|---|
| Uniform | 0.7143 (expected — different edge weighting) |
| Boundary | 0.9995 (boundary grounding dominates) |

## 6. Conclusions

### Recommended backend by use case

| Use Case | Best Backend | Time (1M cells) | Rationale |
|---|---|---|---|
| Uniform absorption, GPU available | **Torch GPU (Jacobi)** | 29 ms | Diagonal dominance → Jacobi is optimal |
| Boundary absorption, GPU available | **JAX GPU (AMJax)** | 222 ms | AMG handles ill-conditioned interior |
| Any strategy, CPU only | **Torch/JAX CPU (pyamg/AMJax)** | 1.6–2.7 s | Both use AMG; similar performance |
| Differentiable optimization | **Torch CPU** or **JAX GPU** | varies | Both support adjoint gradients |

### Why AMGX failed on boundary absorption

AMGX uses **aggregation-based AMG** with a MULTICOLOR_GS smoother. This coarsening strategy
assumes roughly uniform diagonal strength across the matrix. In boundary absorption:

1. **Interior nodes** have diagonal = sum of edge weights (Laplacian only, no absorption boost)
2. **Boundary nodes** have diagonal = sum of edge weights + α

The aggregation algorithm groups interior nodes together, but the coarse grid operator
doesn't capture the fact that current must eventually reach boundary nodes to drain.
Result: PCG needs ~2000 iterations instead of ~30, making AMGX 14× slower than Jacobi
and 63× slower than JAX GPU.

### Key takeaway

**The absorption strategy matters more than the framework.** Boundary absorption is 2–35× slower
than uniform across all backends. The choice of preconditioner determines whether GPU acceleration
helps: Jacobi dominates for uniform (well-conditioned), AMG dominates for boundary (ill-conditioned).
JAX+AMJax is the only backend that handles both strategies efficiently on GPU.

## Appendix: Reproducing

```bash
# In WSL2 with CUDA 12.x
cd /mnt/c/Users/.../DiffiScape/.claude/worktrees/jax-migration
source ~/diffiscape-bench/bin/activate
export LD_LIBRARY_PATH=$HOME/AMGX/build:$HOME/diffiscape-bench/lib/python3.14/site-packages/nvidia/cu13/lib:$LD_LIBRARY_PATH
python bench_full.py
```

Results are saved to `bench_results.json` for programmatic access.